In [9]:
import matplotlib
#matplotlib.use('Agg')
from scipy.spatial.distance import pdist
import matplotlib.pyplot as plt
import os
import numpy as np
from fairchem.core.datasets import AseDBDataset
from ase.calculators.emt import EMT 
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from fairchem.core import pretrained_mlip, FAIRChemCalculator
from ase.io.trajectory import Trajectory
from ase import units
from ase.io import read



In [ ]:

dataset_path = "./train_4M" # using 4m dataset for exploration. Much smaller than the whole dataset
dataset = AseDBDataset({"src": dataset_path})
print(f"Total molecules in this split: {len(dataset)}")

# Extracting dataset info
for i in range(3):
    # This returns a standard ASE Atoms object, but heavily populated by fairchem
    atoms = dataset.get_atoms(i)
    
    print(f"\nMolecule {i+1} Formula: {atoms.get_chemical_formula()}")
    
    print("First 2 atom coordinates (x,y,z):")
    print(atoms.get_positions()[:2]) # can do atoms.positions to get the full list of x,y,z coordinates
    
    #custom OMol25 properties (charges, spins, etc.) are stored in the .info dictionary
    info = atoms.info
    print("\nOMol25 Properties from DATASET.md:")
    if 'omol_energy' in info:
        print(f" - DFT Energy: {info['omol_energy']} eV")
    if 'charge' in info:
        print(f" - System Charge: {info['charge']}")
    if 'spin' in info:
        print(f" - System Spin: {info['spin']}")


Total molecules in this split: 3986754

Molecule 1 Formula: C19H18N4O5S
First 2 atom coordinates (x,y,z):
[[-5.702732e+00  9.472200e-02 -1.235872e+00]
 [-7.180607e+00 -6.575000e-03 -1.062806e+00]]

OMol25 Properties from DATASET.md:
 - System Charge: 0
 - System Spin: 1

Molecule 2 Formula: C2H7O2
First 2 atom coordinates (x,y,z):
[[ 0.281243  1.156284 -2.455943]
 [-0.621103  1.302075 -2.020806]]

OMol25 Properties from DATASET.md:
 - System Charge: -1
 - System Spin: 1

Molecule 3 Formula: C15H18ClINO4V
First 2 atom coordinates (x,y,z):
[[-2.452600e-01 -5.345370e-01  1.700000e-05]
 [-8.897070e-01 -3.335830e-01  3.077262e+00]]

OMol25 Properties from DATASET.md:
 - System Charge: 1
 - System Spin: 4


In [ ]:
#Code below generates molecular motion coordinates for a single molecule
# 1. Load a molecule from your extracted OMol25 dataset
atoms = dataset.get_atoms(1)
print(f"Loaded molecule: {atoms.get_chemical_formula()}")

# 2. Attach the physics engine to the atoms. This will calculate the forces acting on atoms. 
# for production use FAIRChemCalculator, for fast testing use EMT

#atoms.calc = EMT(). Faster engine to test code. 
predictor = pretrained_mlip.get_predict_unit("uma-s-1p2", device="cpu")
calc = FAIRChemCalculator(predictor, task_name="omol")
atoms.calc = calc

# 3. Set the Temperature and give atoms initial velocity
temperature_K = 300  #in Kelvin
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature_K) # this assigns a random initial velocity for each atom given the input temperature

# 4. Set up the MD Integrator
# Langevin dynamics simulates the molecule in a realistic, "messy" environment.
# At each time step it recalculates the position of molecules given the force output by FAIRChemCalulator or EMT and noise from temp and friction. 
time_step = 1.0 * units.fs  # 1 femtosecond per step
friction = 0.01 / units.fs  # Friction coefficient for the heat bath

dyn = Langevin(atoms, time_step, temperature_K=temperature_K, friction=friction)

# 5. Set up the Trajectory Writer
# This will save the (x,y,z) coordinates for each atom in the molecule at each interval step
traj_file = "molecule_motion.traj"
traj = Trajectory(traj_file, 'w', atoms)

# Tell the engine interval between saving coordinates
dyn.attach(traj.write, interval=10) 

print("Starting molecular dynamics simulation...")
steps = 1000 # Run for 1000 femtoseconds (1 picosecond)
dyn.run(steps)
print(f"Simulation complete! Trajectory saved to {traj_file}")

Loaded molecule: C2H7O2


/opt/homebrew/Caskroom/miniconda/base/envs/SLAC/lib/python3.12/site-packages/ase/md/langevin.py:110: FutureWarning: The implementation of `fixcm=True` in `Langevin` does not strictly sample the correct NVT distributions. The deviations are typically small for large systems but can be more pronounced for small systems. Use `fixcm=False` together with `ase.constraints.FixCom`. `fixcm` is deprecated since ASE 3.28.0 and will be removed in a future release.
  warnings.warn(msg, FutureWarning)


Starting molecular dynamics simulation...
Simulation complete! Trajectory saved to molecule_motion.traj


In [ ]:

# Read the entire trajectory into a list of frames
# Each 'frame' is an Atoms object at a specific point in time (contains xyz coordinates for each atom in molecule)
trajectory_frames = read("molecule_motion.traj", index=":") 

print(f"Loaded {len(trajectory_frames)} frames of motion.")

# Get the coordinates at the very first frame (t=0)
coords_t0 = trajectory_frames[0].get_positions()

# Get the coordinates at the last frame
coords_t_end = trajectory_frames[-1].get_positions()

Loaded 101 frames of motion.


In [ ]:
#code below calculates PD and differential PD from the trajectories of a single molecule. 
# 1. Define the distance "radar" parameters (the r-grid)
r_min = 0.0      # Minimum distance in Ångströms. Should remain fixed. 
r_max = 10.0     # Maximum distance in Ångströms (adjust based on molecule size)
num_bins = 100   # Resolution of your X-ray "detector". Determined by the minimum distance delta that can be measured. 
bins = np.linspace(r_min, r_max, num_bins + 1)
r_centers = (bins[:-1] + bins[1:]) / 2 # Center of each bin for plotting

num_frames = len(trajectory_frames)

# Initialize the empty PD matrix (Shape: 101 time steps x 100 distance bins)
PD_matrix = np.zeros((num_frames, num_bins))

print("Calculating Time-Dependent Pair Densities...")

# 2. Build the Time-Dependent PD Matrix
for t in range(num_frames):
    # Get the (x,y,z) coordinates for this specific frame
    positions = trajectory_frames[t].get_positions()
    
    # Calculate all unique pairwise distances instantly
    distances = pdist(positions)
    
    # Bin the distances into a histogram
    counts, _ = np.histogram(distances, bins=bins)
    
    # Store this frame's histogram as a row in our matrix
    PD_matrix[t, :] = counts

# 3. Calculate the Differential PD (Delta PD) Matrix
# Subtract the very first frame (t=0) from every other frame. Can toggle the time delta between PDs
PD_t0 = PD_matrix[0, :]
delta_PD_matrix = PD_matrix - PD_t0

print(f"Delta PD Matrix Shape: {delta_PD_matrix.shape}")

<>:53: SyntaxWarning: invalid escape sequence '\D'
<>:53: SyntaxWarning: invalid escape sequence '\D'
/var/folders/nn/18cpry7128317g4gmfl08t8r0000gn/T/ipykernel_95794/1841959851.py:53: SyntaxWarning: invalid escape sequence '\D'
  axes[1].set_title("Differential PD ($\Delta$PD)")


Calculating Time-Dependent Pair Densities...
Success! Delta PD Matrix Shape: (101, 100)


/var/folders/nn/18cpry7128317g4gmfl08t8r0000gn/T/ipykernel_95794/1841959851.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
#visualization of PDs and differential PDs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Time-Dependent PD
im1 = axes[0].imshow(PD_matrix, aspect='auto', origin='lower', 
                     extent=[r_min, r_max, 0, num_frames*10], cmap='viridis')
axes[0].set_title("Time-Dependent PD")
axes[0].set_xlabel("Distance $r$ (Å)")
axes[0].set_ylabel("Time (femtoseconds)")
fig.colorbar(im1, ax=axes[0], label="Count of Pairs")

# Plot Differential PD (Delta PD)
# Use a diverging colormap (RdBu) because values can be negative or positive
vmax = np.max(np.abs(delta_PD_matrix))
im2 = axes[1].imshow(delta_PD_matrix, aspect='auto', origin='lower', 
                     extent=[r_min, r_max, 0, num_frames*10], cmap='RdBu_r', 
                     vmin=-vmax, vmax=vmax)
axes[1].set_title("Differential PD ($\Delta$PD)")
axes[1].set_xlabel("Distance $r$ (Å)")
axes[1].set_ylabel("Time (femtoseconds)")
fig.colorbar(im2, ax=axes[1], label="Change in Pair Count")

plt.tight_layout()
plt.savefig("pair_density_heatmaps.png") # Uncomment to save if inline plotting fails
